# Допуск: In-the-Flow Agentic System Optimization for Effective Planning and Tool Use (AgentFlow)

**Статья:** Z. Li, H. Zhang, S. Han, S. Liu, J. Xie, Y. Zhang, Y. Choi, J. Zou, P. Lu, 2025 —
<https://arxiv.org/abs/2510.05592>

5 заданий. Под заголовком каждого задания указан его тип:

- **что есть что** — названия функций заменены: определите, что чему соответствует, и
  ответьте на вопросы;
- **расхождение со статьёй** — код реализует не то, что описано в статье: найдите это место;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду: найдите
  его;
- **вопрос по коду** — в коде нет отклонений: ответьте на вопросы, опираясь на код и статью.

Для найденного отклонения напишите: где оно, что не так, какое место статьи (раздел /
формула / таблица / рисунок / цитата) нарушено и как должно быть. Код писать не нужно —
достаточно объяснить словами. Запускать и отлаживать код тоже не нужно: все задания решаются
чтением кода и статьи.

Общие импорты:

In [ ]:
from statistics import mean
import torch

`q` — вопрос, `y` — эталонный ответ, `tools` — набор инструментов K (Base Generator, Python
Coder, Google Search, Wikipedia Search, Web Search). `judge(answer, y)` — LLM-судья,
возвращает 1, если ответ эквивалентен эталону, иначе 0 (прил. E.3). `run_system(planner, q,
tools, T_max)` — один прогон агентной системы с данным планировщиком; возвращает объект с
полями `answer` (итоговый ответ) и `turns` (список пар «состояние — действие планировщика»).
Функции, которые используются, но не показаны, считайте корректными.

## Задание 1. Четыре способа получить агента

**Тип задания:** что есть что — названия функций заменены, определите, что чему соответствует.

Определите, какая функция соответствует: монолитной модели с инструментами, обученной RL
(tool-integrated reasoning, как Search-R1 и ToRL); агентной системе без обучения (как
AutoGen); офлайн-SFT планировщика из табл. 3; AgentFlow с Flow-GRPO. Ошибок в коде нет.

In [ ]:
def train_a(policy, data, tools):
    for q, y in data:
        group = [rollout_interleaved(policy, q, tools) for _ in range(8)]  # <think>, <tool_call>, results
        rewards = [judge(g.answer, y) for g in group]
        grpo_update(policy, [g.tokens for g in group], rewards)            # one sequence, full context

def train_b(planner, data, tools):
    return planner

def train_c(planner, data, tools, teacher):
    traces = [run_system(teacher, q, tools, T_max=10) for q, _ in data]  # teacher = GPT-4o planner
    pairs = [(s, a) for tr in traces for s, a in tr.turns]
    sft_update(planner, pairs)

def train_d(planner, data, tools):
    for q, y in data:
        group = [run_system(planner, q, tools, T_max=3) for _ in range(8)]
        rewards = [judge(g.answer, y) for g in group]
        flow_grpo_update(planner, [g.turns for g in group], rewards)

**Вопросы.**
1. По табл. 3 одна из функций приводит к падению точности ниже исходного замороженного
   планировщика (с 38.5 до 19.5 в среднем), хотя учится у более сильной модели. Какая? Как
   авторы объясняют провал и чем принципиально отличается функция, которая даёт 55.7?
2. В какой функции длина входа модели растёт с каждым вызовом инструмента, а в какой
   планировщик видит компактную память? Какие проблемы монолитного подхода называют авторы
   (§1, рис. 3) и что показывает рис. 9b?
3. В `train_d` награду получает итоговый ответ, а обновляется только планировщик; остальные
   модули — замороженные Qwen2.5-7B-Instruct. Как же планировщик учится хорошо
   пользоваться ими? Почему для этого важно обучение «в потоке», on-policy (§3.2)?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Прогон системы

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Один прогон AgentFlow (§3.1, алгоритм 1). `planner` — обучаемая политика π_θ, `executor_llm`
по плану пишет команду `execution = tool.execute(...)`, `run_command` её исполняет,
`generator_llm` пишет итоговый ответ. Найдите одно отклонение от статьи.

In [ ]:
def run_system(planner, q, tools, T_max=10):
    memory, turns = [], []
    for t in range(1, T_max + 1):
        state = render_state(q, tools, memory)                       # s_t = (q, K, M_t)
        plan = planner(state)                                         # sub-goal, tool, context
        command = executor_llm(q, plan, tools[plan.tool].metadata)
        result = run_command(command, tools[plan.tool])               # e_t
        verdict = "STOP" if result.strip() else "CONTINUE"            # v_t
        memory.append({"turn": t, "tool": plan.tool, "sub_goal": plan.sub_goal,
                       "command": command, "result": result, "verification": verdict})
        turns.append((state, plan.text))
        if verdict == "STOP":
            break
    answer = generator_llm(q, memory)                                 # o ~ G(o | q, M_T)
    return Rollout(answer=answer, turns=turns)

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Advantage для ходов планировщика

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

По группе из G прогонов одного вопроса функция строит advantage для каждого хода каждого
прогона; дальше они идут в клиппированный policy-gradient loss. Найдите одно отклонение от
§3.2.

In [ ]:
def turn_advantages(rewards, n_turns):
    # rewards: G final-outcome rewards in {0, 1} for one query; n_turns: [T_1, ..., T_G]
    r = torch.tensor(rewards, dtype=torch.float)
    adv = (r - r.mean()) / (r.std() + 1e-6)                    # group-normalized
    out = []
    for i, T in enumerate(n_turns):
        per_turn = torch.zeros(T)
        per_turn[-1] = adv[i]                                  # the reward comes after the last turn
        out.append(per_turn)
    return out                                                 # list of (T_i,) tensors

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Нормировка Flow-GRPO

**Тип задания:** вопрос по коду — в коде нет отклонений, ответьте на вопросы.

Клиппированная часть целевой функции Flow-GRPO (формула 5) для одной группы. `logp_new[i][t]`
и `logp_old[i][t]` — log-вероятности токенов действия планировщика на ходе t прогона i
(тензоры длины |a_t^i|), `adv[i][t]` — advantage этого хода.

In [ ]:
def flow_grpo_loss(logp_new, logp_old, adv, eps=0.2):
    G, total = len(logp_new), 0.0
    for i in range(G):
        T_i, traj = len(logp_new[i]), 0.0
        for t in range(T_i):
            ratio = (logp_new[i][t] - logp_old[i][t]).exp()                     # (|a_t|,)
            surr = torch.minimum(ratio * adv[i][t],
                                 ratio.clamp(1 - eps, 1 + eps) * adv[i][t])
            traj = traj + surr.mean()                                          # 1/|a_t| sum over tokens
        total = total + traj / T_i                                             # 1/T_i sum over turns
    return -total / G                       # + beta * KL(pi_theta || pi_ref), beta = 0.001

**Вопросы.**
1. В группе G = 2 прогона: первый — один ход с действием из 50 токенов, второй — три хода
   по 50 токенов. С каким коэффициентом каждый токен каждого прогона входит в `total`?
   Какой суммарный вес у каждого прогона? Как изменились бы веса, если бы loss усредняли по
   всем токенам группы сразу? Чем это было бы плохо, например для прогона, застрявшего в
   цикле из 10 одинаковых неудачных вызовов (рис. 5)?
2. Почему в loss входят только токены планировщика, а не команды исполнителя и не ответы
   инструментов?
3. Теорема B.1 утверждает, что многоходовая задача сводится к последовательности
   «одноходовых» обновлений. Что в этом коде играет роль одного такого обновления? Почему
   при этом не нужна функция ценности (critic), как в PPO с GAE?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Бюджет ходов на инференсе

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

Оценка обученного планировщика при разных максимальных числах ходов (§4.4). `benchmark` —
список пар (q, y). Найдите один неверный комментарий.

In [ ]:
def turn_budget_study(planner, benchmark, tools, budgets=(3, 5, 7, 10), trials=3):
    results = {}
    for T_max in budgets:
        accs, turns = [], []
        for _ in range(trials):                                  # average over three trials
            runs = [run_system(planner, q, tools, T_max=T_max) for q, _ in benchmark]
            accs.append(mean(judge(r.answer, y) for r, (_, y) in zip(runs, benchmark)))
            turns.append(mean(len(r.turns) for r in runs))
        results[T_max] = (mean(accs), mean(turns))
    # The planner was trained with T_max = 3: with a larger budget at test time it falls into
    # repetitive tool-call loops, so accuracy peaks at T_max = 3 and drops for 7 and 10
    return results

**Ваш ответ:**

_Напишите ответ здесь._